In [4]:
import wandb

wandb.login(
    key="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"
)

run = wandb.init(
    project="wandb-test"
)

wandb.log({"loss": 0.5})

run.finish()

print("WandB werkt!")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


loss,▁
loss,0.5


WandB werkt!


In [5]:
import os
import numpy as np
import tensorflow as tf
import wandb
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# =========================
# CONFIG
# =========================
IMG_SIZE = (384, 384)

MODEL_PATHS = {
    "convnext_tiny": "models/best_lizard_model_convnext_tiny.keras",
    "efficientnetb0": "models/best_lizard_model_efficientnetb0.keras",
    "efficientnetv2s": "models/best_lizard_model_efficientnetv2s.keras",
    "ensemble_tta": "models/lizard_weighted_ensemble_flip_tta.keras"
}

# =========================
# WANDB INIT
# =========================
wandb.login()

run = wandb.init(
    project="lizard-model-comparison",
    name="SAFE-full-evaluation"
)

# =========================
# TRAIN (labels)
# =========================
train_ds = tf.keras.utils.image_dataset_from_directory(
    "train",
    image_size=IMG_SIZE,
    batch_size=32,
    shuffle=True
)

class_names = train_ds.class_names
print("Classes:", class_names)

# =========================
# TEST LOADER (KEEP AS IS)
# =========================
def load_test_images(test_dir):
    images, filenames = [], []

    for f in os.listdir(test_dir):
        if f.lower().endswith((".jpg", ".png", ".jpeg")):

            path = os.path.join(test_dir, f)

            img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)
            img = tf.keras.utils.img_to_array(img) / 255.0

            images.append(img)
            filenames.append(f)

    return np.array(images), filenames

test_images, test_files = load_test_images("test")

# =========================
# SAFE MODEL LOADER
# =========================
def load_model_safe(path):
    try:
        if not os.path.exists(path):
            print(f"❌ Missing: {path}")
            return None

        model = tf.keras.models.load_model(path, compile=False)
        print(f"✅ Loaded: {path}")
        return model

    except Exception as e:
        print(f"❌ Error loading {path}")
        print(e)
        return None

# =========================
# STORAGE
# =========================
all_predictions = {}
all_probs = {}
all_accuracies = {}

# =========================
# RUN MODELS (UNCHANGED CORE)
# =========================
for name, path in MODEL_PATHS.items():

    print(f"\nRunning: {name}")

    model = load_model_safe(path)
    if model is None:
        continue

    probs = model.predict(test_images, verbose=0)
    preds = np.argmax(probs, axis=1)

    all_predictions[name] = preds
    all_probs[name] = probs

    # =========================
    # ACCURACY
    # =========================
    acc = accuracy_score(
        np.repeat(0, len(preds)),  # placeholder fix below
        np.zeros(len(preds))       # TEMP (see note)
    )

Found 1178 files belonging to 7 classes.
Classes: ['Black_spiny_tailed_iguana', 'Brown_anole', 'Cuban_knight_anole', 'Desert_iguana', 'Green_anole', 'Green_iguana', 'Lesser_Antillean_iguana']

Running: convnext_tiny

✅ Loaded: models/best_lizard_model_convnext_tiny.keras

Running: efficientnetb0
✅ Loaded: models/best_lizard_model_efficientnetb0.keras

Running: efficientnetv2s
✅ Loaded: models/best_lizard_model_efficientnetv2s.keras

Running: ensemble_tta
❌ Error loading models/lizard_weighted_ensemble_flip_tta.keras
<class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': None}.

Exception encountered: 

In [3]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import wandb
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report
)

from sklearn.decomposition import PCA

# =========================
# CONFIG
# =========================
IMG_SIZE = (384, 384)

MODEL_PATHS = {
    "convnext_tiny": "models/best_lizard_model_convnext_tiny.keras",
    "efficientnetb0": "models/best_lizard_model_efficientnetb0.keras",
    "efficientnetv2s": "models/best_lizard_model_efficientnetv2s.keras",
    "ensemble_tta": "models/lizard_weighted_ensemble_flip_tta.keras"
}

# =========================
# WANDB INIT
# =========================
wandb.login()

run = wandb.init(
    project="lizard-model-comparison",
    name="SAFE-full-evaluation"
)

# =========================
# LOAD TRAIN DATASET
# =========================
train_ds = tf.keras.utils.image_dataset_from_directory(
    "train",
    image_size=IMG_SIZE,
    batch_size=32,
    shuffle=True
)

class_names = train_ds.class_names

print("\nClasses:")
print(class_names)

# =========================
# CHECK CLASS ORDER
# =========================
expected_classes = [
    "Black_spiny_tailed_iguana",
    "Brown_anole",
    "Cuban_knight_anole",
    "Desert_iguana",
    "Green_anole",
    "Green_iguana",
    "Lesser_Antillean_iguana"
]

if class_names != expected_classes:
    print("\n⚠ WARNING: Class order mismatch!")

# =========================
# LOAD TEST IMAGES
# =========================
def load_test_images(test_dir):

    images = []
    filenames = []

    files = sorted(
        os.listdir(test_dir),
        key=lambda x: int(os.path.splitext(x)[0])
    )

    for f in files:
        if f.lower().endswith((".jpg", ".png", ".jpeg")):

            path = os.path.join(test_dir, f)

            img = tf.keras.utils.load_img(
                path,
                target_size=IMG_SIZE
            )

            img = tf.keras.utils.img_to_array(img) / 255.0

            images.append(img)
            filenames.append(f)

    return np.array(images), filenames

test_images, test_files = load_test_images("test")

print("\nAantal test images:", len(test_images))

# =========================
# LOAD TEST LABELS
# =========================
labels_df = pd.read_csv("submission_transfer_learning.csv")
test_labels = labels_df["label"].values

print("Aantal test labels:", len(test_labels))

assert len(test_images) == len(test_labels), "Mismatch images vs labels!"

# =========================
# MODEL LOADER
# =========================
def load_model_safe(path):

    try:
        if not os.path.exists(path):
            print(f"❌ Missing: {path}")
            return None

        model = tf.keras.models.load_model(path, compile=False)

        print(f"✅ Loaded: {path}")
        return model

    except Exception as e:
        print(e)
        return None

# =========================
# STORAGE
# =========================
all_predictions = {}
all_probs = {}
all_accuracies = {}

# =========================
# MAIN LOOP
# =========================
for name, path in MODEL_PATHS.items():

    print(f"\nRunning: {name}")

    model = load_model_safe(path)

    if model is None:
        continue

    try:

        # =========================
        # PREDICT
        # =========================
        probs = model.predict(test_images, verbose=0)

        if len(probs.shape) == 1:
            probs = np.expand_dims(probs, axis=1)

        preds = np.argmax(probs, axis=1)

        all_predictions[name] = preds
        all_probs[name] = probs

        # =========================
        # ENTROPY (UNCERTAINTY)
        # =========================
        entropy = -np.sum(probs * np.log(probs + 1e-9), axis=1)

        wandb.log({
            f"{name}_avg_entropy": float(np.mean(entropy))
        })

        # =========================
        # CONFIDENCE
        # =========================
        confidence = np.max(probs, axis=1)

        wandb.log({
            f"{name}_avg_confidence": float(np.mean(confidence))
        })

        # =========================
        # CONFIDENCE SPLIT
        # =========================
        correct_conf = confidence[preds == test_labels]
        wrong_conf = confidence[preds != test_labels]

        wandb.log({
            f"{name}_avg_correct_confidence": float(np.mean(correct_conf)),
            f"{name}_avg_wrong_confidence": float(np.mean(wrong_conf))
        })

        # =========================
        # ACCURACY
        # =========================
        acc = accuracy_score(test_labels, preds)

        all_accuracies[name] = acc

        wandb.log({
            f"{name}_accuracy": acc
        })

        # =========================
        # CONFUSION MATRIX
        # =========================
        wandb.log({
            f"{name}_conf_matrix":
                wandb.plot.confusion_matrix(
                    probs=None,
                    y_true=test_labels,
                    preds=preds,
                    class_names=class_names
                )
        })

        # =========================
        # TOP CONFUSIONS
        # =========================
        cm = confusion_matrix(test_labels, preds)
        np.fill_diagonal(cm, 0)

        top_confusions = []

        for i in range(len(class_names)):
            for j in range(len(class_names)):
                if cm[i, j] > 0:
                    top_confusions.append((cm[i, j], i, j))

        top_confusions.sort(reverse=True)

        wandb.log({
            f"{name}_top_confusions":
                wandb.Table(
                    columns=["true", "pred", "count"],
                    data=[
                        [class_names[i], class_names[j], int(c)]
                        for c, i, j in top_confusions[:10]
                    ]
                )
        })

        # =========================
        # TOP 10 CONFIDENT WRONG
        # =========================
        wrong_idx = np.where(preds != test_labels)[0]
        wrong_sorted = wrong_idx[np.argsort(-confidence[wrong_idx])]
        top_wrong = wrong_sorted[:10]

        wrong_images = []

        for i in top_wrong:

            img = (test_images[i] * 255).astype(np.uint8)

            wrong_images.append(
                wandb.Image(
                    img,
                    caption=f"Pred: {class_names[preds[i]]} | True: {class_names[test_labels[i]]} | Conf: {confidence[i]:.3f}"
                )
            )

        wandb.log({
            f"{name}_top10_confident_wrong": wrong_images
        })

        # =========================
        # CLASSIFICATION REPORT
        # =========================
        report = classification_report(
            test_labels,
            preds,
            target_names=class_names,
            output_dict=True,
            zero_division=0
        )

        wandb.log({
            f"{name}_per_class_metrics":
                wandb.Table(
                    columns=["class", "precision", "recall", "f1"],
                    data=[
                        [
                            cls,
                            report[cls]["precision"],
                            report[cls]["recall"],
                            report[cls]["f1-score"]
                        ]
                        for cls in class_names
                    ]
                )
        })

    except Exception as e:
        print(f"Error in {name}")
        print(e)

# =========================
# MODEL RANKING
# =========================
ranking = sorted(all_accuracies.items(), key=lambda x: x[1], reverse=True)

wandb.log({
    "model_ranking":
        wandb.Table(
            columns=["model", "accuracy"],
            data=ranking
        )
})

# =========================
# DISAGREEMENT
# =========================
disagreement = []

for i in range(len(test_images)):

    preds_list = [all_predictions[m][i] for m in all_predictions]

    most_common = max(set(preds_list), key=preds_list.count)

    agreement = preds_list.count(most_common) / len(preds_list)

    disagreement.append(1 - agreement)

wandb.log({
    "avg_model_disagreement": float(np.mean(disagreement))
})

# =========================
# BEST MODEL PER IMAGE
# =========================
best_models = []

for i in range(len(test_images)):

    best = "none"

    for m in all_predictions:

        if all_predictions[m][i] == test_labels[i]:
            best = m
            break

    best_models.append(best)

wandb.log({
    "best_model_per_image":
        wandb.Table(
            columns=["image", "best_model"],
            data=[[f"img_{i}", best_models[i]] for i in range(len(best_models))]
        )
})

# =========================
# EMBEDDINGS
# =========================
try:

    best_model_name = max(all_accuracies, key=all_accuracies.get)

    embedding_model = load_model_safe(MODEL_PATHS[best_model_name])

    feature_extractor = tf.keras.Model(
        inputs=embedding_model.input,
        outputs=embedding_model.layers[-2].output
    )

    embeddings = feature_extractor.predict(test_images, verbose=0)
    embeddings = embeddings.reshape(embeddings.shape[0], -1)

    pca = PCA(n_components=2)
    emb_2d = pca.fit_transform(embeddings)

    wandb.log({
        "embeddings_2d":
            wandb.Table(
                columns=["x", "y", "label"],
                data=[
                    [float(emb_2d[i, 0]), float(emb_2d[i, 1]), class_names[test_labels[i]]]
                    for i in range(len(emb_2d))
                ]
            )
    })

except Exception as e:
    print("Embedding failed:", e)

# =========================
# FINISH
# =========================
run.finish()

print("DONE")

Found 1178 files belonging to 7 classes.

Classes:
['Black_spiny_tailed_iguana', 'Brown_anole', 'Cuban_knight_anole', 'Desert_iguana', 'Green_anole', 'Green_iguana', 'Lesser_Antillean_iguana']

Aantal test images: 326
Aantal test labels: 326

Running: convnext_tiny
✅ Loaded: models/best_lizard_model_convnext_tiny.keras

Running: efficientnetb0
✅ Loaded: models/best_lizard_model_efficientnetb0.keras

Running: efficientnetv2s
✅ Loaded: models/best_lizard_model_efficientnetv2s.keras

Running: ensemble_tta
<class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': None}.

Exception encountered: Could not loca

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


avg_model_disagreement,▁
convnext_tiny_accuracy,▁
convnext_tiny_avg_confidence,▁
convnext_tiny_avg_correct_confidence,▁
convnext_tiny_avg_entropy,▁
convnext_tiny_avg_wrong_confidence,▁
efficientnetb0_accuracy,▁
efficientnetb0_avg_confidence,▁
efficientnetb0_avg_correct_confidence,▁
efficientnetb0_avg_entropy,▁
+6,...


DONE
